In [ ]:
#pip install pandas
#pip install sweetviz
#pip install seaborn 
#pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
RAIZ  = Path.cwd().parent
IVTA = RAIZ / "data" / "IVTA" / "informe__IVTA_Distritos_2020-2024.csv"
ivta_df = pd.read_csv(IVTA, sep=";")
mediaMadrid_ivta = ivta_df["Indice de Vulnerabilidad Territorial Agregado - media ciudad"]
mediaMadrid_Vulnerabilidad_Bienestar = ivta_df["Índice de Vulnerabilidad Bienestar Social e Igualdad - media ciudad"]
ivta_df = ivta_df[["Nombre distrito", "Fecha datos", "Indice de Vulnerabilidad Territorial Agregado", 
                   "Índice de Vulnerabilidad Bienestar Social e Igualdad"]]

ivta_df = ivta_df[ivta_df["Nombre distrito"].isin(["Centro", "Villaverde", "Tetuán"])]

ivta_df["Indice de Vulnerabilidad Territorial Agregado"] = ivta_df["Indice de Vulnerabilidad Territorial Agregado"].str.replace(",", ".").astype(float)
ivta_df["Índice de Vulnerabilidad Bienestar Social e Igualdad"] = ivta_df["Índice de Vulnerabilidad Bienestar Social e Igualdad"].str.replace(",", ".").astype(float)

ivta_df = ivta_df.rename(columns={
    "Nombre distrito": "Distrito",
    "Fecha datos": "Año",
})
ivta_df = ivta_df[ivta_df["Distrito"] != "Ciudad de Madrid"]

ivta_df.info()

<class 'pandas.DataFrame'>
Index: 15 entries, 0 to 100
Data columns (total 4 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Distrito                                              15 non-null     str    
 1   Año                                                   15 non-null     int64  
 2   Indice de Vulnerabilidad Territorial Agregado         15 non-null     float64
 3   Índice de Vulnerabilidad Bienestar Social e Igualdad  15 non-null     float64
dtypes: float64(2), int64(1), str(1)
memory usage: 600.0 bytes


In [3]:
precioVivienda = RAIZ / "data" / "precio vivienda" / "Ayuntamiento_vivienda_distrito_m2_2020-2025.csv"
precioVivienda_df = pd.read_csv(precioVivienda, sep=";", skiprows=5, thousands=".", decimal=",")
precioVivienda_df = precioVivienda_df[["Año", "Distrito", "Euros/m2"]]

precioVivienda_df = precioVivienda_df[precioVivienda_df["Distrito"].isin(["01. Centro", "17. Villaverde", "06. Tetuán"])]
precioVivienda_df = precioVivienda_df[precioVivienda_df["Año"].isin([2020, 2021, 2022, 2023, 2024])]
precioVivienda_df["Año"] = precioVivienda_df["Año"].astype(int)
precioVivienda_df["Distrito"] = precioVivienda_df["Distrito"].str.replace(r"^\d+\.\s*", "", regex=True)

precioVivienda_df = precioVivienda_df[precioVivienda_df["Distrito"] != "Ciudad de Madrid"]

precioVivienda_df = precioVivienda_df.rename(columns={
    "Euros/m2": "Precio vivienda (€/m²)",
})
precioVivienda_df.info()

<class 'pandas.DataFrame'>
Index: 15 entries, 1 to 105
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Año                     15 non-null     int64  
 1   Distrito                15 non-null     str    
 2   Precio vivienda (€/m²)  15 non-null     float64
dtypes: float64(1), int64(1), str(1)
memory usage: 480.0 bytes


In [20]:
precioAlquiler = RAIZ / "data" / "precio vivienda - alquiler" / "Precios_alquiler_estadísticas_año-distrito.csv"
precioAlquiler_df = pd.read_csv(precioAlquiler, sep=";", skiprows=4)
precioAlquiler_df["Año"] = pd.to_numeric(precioAlquiler_df["Año"], errors="coerce").astype("Int64")

cols_int = ["Número de viviendas en alquiler", "Número de viviendas en alquiler.1", "Número de viviendas en alquiler.2"]

for col in cols_int:
    precioAlquiler_df[col] = precioAlquiler_df[col].str.replace(".", "", regex=False)
    precioAlquiler_df[col] = pd.to_numeric(
        precioAlquiler_df[col].str.replace(",", ".", regex=False),
        errors="coerce"
    )

for col in cols_int:
    precioAlquiler_df[col] = pd.to_numeric(precioAlquiler_df[col], errors="coerce")

precioAlquiler_df = precioAlquiler_df.dropna(subset=cols_int)

precioAlquiler_df[cols_int] = precioAlquiler_df[cols_int].astype(int)

cols_float = ["Renta media vivienda colectiva (€/m2.mes)", "Renta media vivienda unifamiliar (€/m2.mes)"]

for col in cols_float:
    precioAlquiler_df[col] = precioAlquiler_df[col].str.replace(".", "", regex=False)
    precioAlquiler_df[col] = pd.to_numeric(
        precioAlquiler_df[col].str.replace(",", ".", regex=False),
        errors="coerce"
    )

precioAlquiler_df = precioAlquiler_df[precioAlquiler_df["Distrito"].isin(["01. Centro", "17. Villaverde", "06. Tetuán"])]

precioAlquiler_df = precioAlquiler_df[["Año", "Distrito", "Número de viviendas en alquiler", "Número de viviendas en alquiler.1", "Número de viviendas en alquiler.2",
                                       "Renta media vivienda colectiva (€/m2.mes)", "Renta media vivienda unifamiliar (€/m2.mes)"]]

precioAlquiler_df["Distrito"] = precioAlquiler_df["Distrito"].str.replace(r"^\d+\.\s*", "", regex=True)

precioAlquiler_df = precioAlquiler_df[precioAlquiler_df["Distrito"] != "Ciudad de Madrid"]


precioAlquiler_df = precioAlquiler_df.rename(columns={
    "Número de viviendas en alquiler": "Nº viviendas en alquiler",
    "Número de viviendas en alquiler.1": "Nº viviendas en alquiler colectiva",
    "Número de viviendas en alquiler.2": "Nº viviendas en alquiler unifamiliar",
    "Renta media vivienda colectiva (€/m2.mes)": "Renta vivienda colectiva (€/m².mes)",
    "Renta media vivienda unifamiliar (€/m2.mes)": "Renta vivienda unifamiliar (€/m².mes)",
})

precioAlquiler_df.head()


,Año,Distrito,Nº viviendas en alquiler,Nº viviendas en alquiler colectiva,Nº viviendas en alquiler unifamiliar,Renta vivienda colectiva (€/m².mes),Renta vivienda unifamiliar (€/m².mes)
2,2020,Centro,18981,18974,7,15.36,0.00
7,2020,Tetuán,17586,17535,51,13.82,11.44
18,2020,Villaverde,7956,7887,69,8.65,8.12
24,2021,Centro,19583,19576,7,15.32,0.00
29,2021,Tetuán,18140,18082,58,13.95,11.62


In [21]:
precioAlquiler_meses = RAIZ / "data" / "precio vivienda - alquiler" / "precio_alquiler_distrito_mes.csv"
precioAlquiler_meses_df = pd.read_csv(precioAlquiler_meses, sep=",", skiprows=4)

precioAlquiler_meses_df = precioAlquiler_meses_df[precioAlquiler_meses_df["Distrito"].isin(["01. Centro", "17. Villaverde", "06. Tetuán"])]

cols_float = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]

for col in cols_float:
    precioAlquiler_meses_df[col] = pd.to_numeric(
        precioAlquiler_meses_df[col].str.replace(",", ".", regex=False),
        errors="coerce"
    )


precioAlquiler_meses_df["Precio alquiler €/m² media del año"] = precioAlquiler_meses_df[["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]].mean(axis=1)

precioAlquiler_meses_df = precioAlquiler_meses_df[["Año", "Distrito", "Precio alquiler €/m² media del año"]]

precioAlquiler_meses_df["Año"] = precioAlquiler_meses_df["Año"].astype(int)
precioAlquiler_meses_df["Distrito"] = precioAlquiler_meses_df["Distrito"].str.replace(r"^\d+\.\s*", "", regex=True)
precioAlquiler_meses_df = precioAlquiler_meses_df[precioAlquiler_meses_df["Distrito"] != "Ciudad de Madrid"]

precioAlquiler_meses_df.head()

,Año,Distrito,Precio alquiler €/m² media del año
1,2020,Centro,18.158333
6,2020,Tetuán,15.750000
17,2020,Villaverde,10.950000
23,2021,Centro,16.375000
28,2021,Tetuán,14.425000


In [14]:
apartamentosTuristicos = RAIZ / "data" / "apartamentos turisticos" / "apartamentos turísticos_2020-2024_distri_ establecimientos-plazas.csv"
apartamentosTuristicos_df = pd.read_csv(apartamentosTuristicos, sep=";", skiprows=4, thousands=".")

apartamentosTuristicos_df = apartamentosTuristicos_df[apartamentosTuristicos_df["Distrito"].isin(["01. Centro", "17. Villaverde", "06. Tetuán"])]

apartamentosTuristicos_df = apartamentosTuristicos_df[["Año", "Distrito", "Establecimientos", "Plazas"]]

apartamentosTuristicos_df["Año"] = apartamentosTuristicos_df["Año"].astype(int)
apartamentosTuristicos_df["Establecimientos"] = apartamentosTuristicos_df["Establecimientos"].astype(int)
apartamentosTuristicos_df["Plazas"] = apartamentosTuristicos_df["Plazas"].astype(int)
apartamentosTuristicos_df["Distrito"] = apartamentosTuristicos_df["Distrito"].str.replace(r"^\d+\.\s*", "", regex=True)

apartamentosTuristicos_df = apartamentosTuristicos_df[apartamentosTuristicos_df["Año"].isin([2020, 2021, 2022, 2023, 2024])]


apartamentosTuristicos_df = apartamentosTuristicos_df.rename(columns={
    "Establecimientos": "Establecimientos turisticos",
    "Plazas": "Plazas en sstablecimientos turisticos",
})

apartamentosTuristicos_df = apartamentosTuristicos_df[apartamentosTuristicos_df["Distrito"] != "Ciudad de Madrid"]

apartamentosTuristicos_df.head()


,Año,Distrito,Establecimientos turisticos,Plazas en sstablecimientos turisticos
1,2020,Centro,48,2643
6,2020,Tetuán,4,301
17,2020,Villaverde,1,1
23,2021,Centro,55,3096
28,2021,Tetuán,5,327


In [26]:
poblacion = RAIZ / "data" / "poblacion distrito" / "poblacion-distrito-barrio_2018-2024.csv"
poblacion_df = pd.read_csv(poblacion, sep=";", thousands=".")

poblacion_df["barrio"] = poblacion_df["barrio"].str.strip()
poblacion_df["distrito"] = poblacion_df["distrito"].str.strip()


poblacion_df = poblacion_df[poblacion_df["distrito"] == poblacion_df["barrio"]]
poblacion_df = poblacion_df[poblacion_df["barrio"].isin(["Centro", "Villaverde", "Tetuán"])]

poblacion_df["Año"] = poblacion_df["fecha"].str.extract(r"(\d{4})").astype(int)
poblacion_df = poblacion_df[poblacion_df["distrito"] == poblacion_df["barrio"]]

poblacion_df = poblacion_df.rename(columns={
    "distrito": "Distrito",
    "num_personas": "Poblacion",
})
poblacion_df["Distrito"] = poblacion_df["Distrito"].replace("Todos", "Ciudad de Madrid")

poblacion_df = poblacion_df[["Año", "Distrito", "Poblacion"]]
poblacion_df = poblacion_df[poblacion_df["Año"].isin([2020, 2021, 2022, 2023, 2024])]

poblacion_df = poblacion_df[poblacion_df["Distrito"] != "Ciudad de Madrid"]

poblacion_df.head(21)
#print(poblacion_df.to_string())

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\danie\\OneDrive\\Escritorio\\UNIR\\TFM\\data\\poblacion distrito\\poblacion-distrito-barrio_2018-2024.csv'

In [23]:
sns.lineplot(
    data=apartamentosTuristicos_df,
    x="Año",
    y="Plazas en sstablecimientos turisticos-",
    hue="Distrito",
    marker="o"
)

plt.title("Evolución del IVTA por distrito")
plt.show()

ValueError: Could not interpret value `Plazas en sstablecimientos turisticos-` for `y`. An entry with this name does not appear in `data`.

In [27]:
sns.lineplot(
    data=precioAlquiler_meses_df,
    x="Año",
    y="Precio alquiler €/m² media del año-",
    hue="Distrito",
    marker="o"
)

plt.title("Evolución del precio del alquiler por distrito")
plt.show()

ValueError: Could not interpret value `Precio alquiler €/m² media del año-` for `y`. An entry with this name does not appear in `data`.

In [9]:
#apartamentosTuristicos_df, encuestaOcupacion_df, precioAlquiler_df, precioVivienda_df, ivta_df
df_total = (
    apartamentosTuristicos_df
    .merge(precioAlquiler_df, on=["Año", "Distrito"], how="inner")
    .merge(precioVivienda_df, on=["Año", "Distrito"], how="inner")
    .merge(ivta_df, on=["Año", "Distrito"], how="inner")
    .merge(precioAlquiler_meses_df, on=["Año", "Distrito"], how="inner")
    .merge(poblacion_df, on=["Año", "Distrito"], how="inner")
)


#df_total = pd.concat([apartamentosTuristicos_df, precioAlquiler_df, precioVivienda_df, ivta_df], axis=1)

corr = df_total.corr(numeric_only=True)
#print(corr)
df_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 14 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Año                                                   15 non-null     int64  
 1   Distrito                                              15 non-null     str    
 2   Establecimientos turisticos                           15 non-null     int64  
 3   Plazas en sstablecimientos turisticos                 15 non-null     int64  
 4   Nº viviendas en alquiler                              15 non-null     int64  
 5   Nº viviendas en alquiler colectiva                    15 non-null     int64  
 6   Nº viviendas en alquiler unifamiliar                  15 non-null     int64  
 7   Renta vivienda colectiva (€/m².mes)                   15 non-null     float64
 8   Renta vivienda unifamiliar (€/m².mes)                 15 non-null     flo

In [10]:
import sweetviz as sv

reporte = sv.analyze(df_total)
reporte.show_html("reporte_correlaciones.html")

c:\Users\danie\OneDrive\Escritorio\UNIR\TFM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Feature: Año                                 |▋         | [  7%]   00:00 -> (00:00 left)findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
Feature: Distrito                            |█▎        | [ 13%]   00:00 -> (00:01 left)findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
Feature: Establecimientos turisticos         |██        | [ 20%]   00:0

Report reporte_correlaciones.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [ ]:
variables = [
    "Indice de Vulnerabilidad Territorial Agregado",
    "Índice de Vulnerabilidad Bienestar Social e Igualdad",
    "Precio vivienda (€/m²)",
    "Renta vivienda unifamiliar (€/m².mes)",
    "Renta vivienda colectiva (€/m².mes)",
    "Nº viviendas en alquiler",
    "Establecimientos turisticos",
    "Plazas en sstablecimientos turisticos",
    "Poblacion"
]

corr_pearson = df_total[variables].corr(method="pearson")

print("Pearson")
print(corr_pearson.round(2))

Pearson
                                                    Indice de Vulnerabilidad Territorial Agregado  \
Indice de Vulnerabilidad Territorial Agregado                                                1.00   
Índice de Vulnerabilidad Bienestar Social e Igu...                                           0.71   
Precio vivienda (€/m²)                                                                      -0.42   
Renta vivienda unifamiliar (€/m².mes)                                                       -0.20   
Renta vivienda colectiva (€/m².mes)                                                         -0.35   
Nº viviendas en alquiler                                                                     0.09   
Establecimientos turisticos                                                                  0.18   
Plazas en sstablecimientos turisticos                                                        0.08   
num_personas                                                                       

In [283]:
corr_spearman = df_total[variables].corr(method="spearman")

print("Spearman")
print(corr_spearman.round(2))

Spearman
                                                    Indice de Vulnerabilidad Territorial Agregado  \
Indice de Vulnerabilidad Territorial Agregado                                                1.00   
Índice de Vulnerabilidad Bienestar Social e Igu...                                           0.77   
Precio vivienda (€/m²)                                                                      -0.04   
Renta vivienda unifamiliar (€/m².mes)                                                       -0.70   
Renta vivienda colectiva (€/m².mes)                                                         -0.11   
Nº viviendas en alquiler                                                                    -0.14   
Establecimientos turisticos                                                                 -0.05   
Plazas en sstablecimientos turisticos                                                       -0.06   
num_personas                                                                      

In [284]:
df_total = df_total.sort_values(["Distrito", "Año"])

cols_evolucion = [
    "Establecimientos turisticos",
    "Plazas en sstablecimientos turisticos",
    "Precio vivienda (€/m²)",
    "Precio alquiler €/m² media del año",
    "Indice de Vulnerabilidad Territorial Agregado",
    "Índice de Vulnerabilidad Bienestar Social e Igualdad"
]

for col in cols_evolucion:
    df_total[col + "_var_pct"] = df_total.groupby("Distrito")[col].pct_change() * 100

df_total.head(30)

,Año,Distrito,Establecimientos turisticos,Plazas en sstablecimientos turisticos,Nº viviendas en alquiler,Nº viviendas en alquiler colectiva,Nº viviendas en alquiler unifamiliar,Renta vivienda colectiva (€/m².mes),Renta vivienda unifamiliar (€/m².mes),Precio vivienda (€/m²),Indice de Vulnerabilidad Territorial Agregado,Índice de Vulnerabilidad Bienestar Social e Igualdad,Precio alquiler €/m² media del año,num_personas,Establecimientos turisticos_var_pct,Plazas en sstablecimientos turisticos_var_pct,Precio vivienda (€/m²)_var_pct,Precio alquiler €/m² media del año_var_pct,Indice de Vulnerabilidad Territorial Agregado_var_pct,Índice de Vulnerabilidad Bienestar Social e Igualdad_var_pct
0,2020,Centro,48,2643,18981,18974,7,15.36,0.00,5223.73,5.53,5.98,18.158333,140473,NaN,NaN,NaN,NaN,NaN,NaN
3,2021,Centro,55,3096,19583,19576,7,15.32,0.00,4935.90,5.57,6.56,16.375000,141236,14.583333,17.139614,-5.510047,-9.821019,0.723327,9.698997
6,2022,Centro,58,3051,19945,19938,7,16.01,0.00,5466.32,5.78,6.61,18.225000,139682,5.454545,-1.453488,10.746166,11.297710,3.770197,0.762195
9,2023,Centro,59,3109,20393,20385,8,16.77,0.00,5712.09,5.66,6.17,20.583333,139687,1.724138,1.901016,4.496078,12.940101,-2.076125,-6.656581
12,2024,Centro,59,3268,20525,20518,7,17.70,0.00,6268.72,5.89,6.45,23.791667,145411,0.000000,5.114185,9.744769,15.587045,4.063604,4.538088
1,2020,Tetuán,4,301,17586,17535,51,13.82,11.44,3648.00,5.34,4.53,15.750000,161313,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,Tetuán,5,327,18140,18082,58,13.95,11.62,3756.93,5.24,4.48,14.425000,159849,25.000000,8.637874,2.986020,-8.412698,-1.872659,-1.103753
7,2022,Tetuán,6,357,18764,18703,61,14.52,12.20,3955.42,4.89,4.09,15.658333,157433,20.000000,9.174312,5.283303,8.549971,-6.679389,-8.705357
10,2023,Tetuán,9,399,19616,19549,67,15.22,12.52,4677.48,4.81,3.86,17.383333,160002,50.000000,11.764706,18.254951,11.016498,-1.635992,-5.623472
13,2024,Tetuán,15,637,20337,20271,66,16.08,13.39,4755.80,4.69,3.78,19.825000,166211,66.666667,59.649123,1.674406,14.046021,-2.494802,-2.072539


In [178]:
resumen = df_total.groupby("Distrito").agg({
    "Establecimientos turisticos": ["first", "last"],
    "Plazas en sstablecimientos turisticos": ["first", "last"],
    "Precio vivienda (€/m²)": ["first", "last"],
    "Precio alquiler €/m² media del año": ["first", "last"],
    "Indice de Vulnerabilidad Territorial Agregado": ["first", "last"],
    "Índice de Vulnerabilidad Bienestar Social e Igualdad": ["first", "last"]
})

print(resumen)

           Establecimientos turisticos       \
                                 first last   
Distrito                                      
Centro                              48   59   
Tetuán                               4   15   
Villaverde                           1    3   

           Plazas en sstablecimientos turisticos       Precio vivienda (€/m²)  \
                                           first  last                  first   
Distrito                                                                        
Centro                                      2643  3268                5223.73   
Tetuán                                       301   637                3648.00   
Villaverde                                     1    28                1749.87   

                    Precio alquiler €/m² media del año             \
               last                              first       last   
Distrito                                                            
Centro      6268.72    

In [179]:
df_2020 = df_total[df_total["Año"] == 2020].set_index("Distrito")
df_2024 = df_total[df_total["Año"] == 2024].set_index("Distrito")

variables = [
    "Establecimientos turisticos",
    "Plazas en sstablecimientos turisticos",
    "Precio vivienda (€/m²)",
    "Precio alquiler €/m² media del año",
    "Indice de Vulnerabilidad Territorial Agregado",
    "Índice de Vulnerabilidad Bienestar Social e Igualdad"
]

variacion_2020_2024 = ((df_2024[variables] - df_2020[variables]) / df_2020[variables]) * 100

print(variacion_2020_2024.round(2))

            Establecimientos turisticos  \
Distrito                                  
Centro                            22.92   
Tetuán                           275.00   
Villaverde                       200.00   

            Plazas en sstablecimientos turisticos  Precio vivienda (€/m²)  \
Distrito                                                                    
Centro                                      23.65                   20.00   
Tetuán                                     111.63                   30.37   
Villaverde                                2700.00                   20.74   

            Precio alquiler €/m² media del año  \
Distrito                                         
Centro                                   31.02   
Tetuán                                   25.87   
Villaverde                               33.49   

            Indice de Vulnerabilidad Territorial Agregado  \
Distrito                                                    
Centro                   

In [28]:


sns.lineplot(
    data=df_total,
    x="Año",
    y="Precio alquiler €/m² media del año-",
    hue="Distrito",
    marker="o"
)

plt.title("Evolución del precio del alquiler por distrito")
plt.show()

ValueError: Could not interpret value `Precio alquiler €/m² media del año-` for `y`. An entry with this name does not appear in `data`.

In [29]:
sns.lineplot(
    data=df_total,
    x="Año",
    y="Precio vivienda (€/m²)-",
    hue="Distrito",
    marker="o"
)

plt.title("Evolución del precio de vivienda por distrito")
plt.show()

ValueError: Could not interpret value `Precio vivienda (€/m²)-` for `y`. An entry with this name does not appear in `data`.

In [30]:
sns.lineplot(
    data=df_total,
    x="Año",
    y="Indice de Vulnerabilidad Territorial Agregado-",
    hue="Distrito",
    marker="o"
)

plt.title("Evolución del IVTA por distrito")
plt.show()

ValueError: Could not interpret value `Indice de Vulnerabilidad Territorial Agregado-` for `y`. An entry with this name does not appear in `data`.

In [187]:
df_total

,Año,Distrito,Establecimientos turisticos,Plazas en sstablecimientos turisticos,Nº viviendas en alquiler,Nº viviendas en alquiler colectiva,Nº viviendas en alquiler unifamiliar,Renta vivienda colectiva (€/m².mes),Renta vivienda unifamiliar (€/m².mes),Precio vivienda (€/m²),Indice de Vulnerabilidad Territorial Agregado,Índice de Vulnerabilidad Bienestar Social e Igualdad,Precio alquiler €/m² media del año,Establecimientos turisticos_var_pct,Plazas en sstablecimientos turisticos_var_pct,Precio vivienda (€/m²)_var_pct,Precio alquiler €/m² media del año_var_pct,Indice de Vulnerabilidad Territorial Agregado_var_pct,Índice de Vulnerabilidad Bienestar Social e Igualdad_var_pct
0,2020,Centro,48,2643,18981,18974,7,15.36,0.00,5223.73,5.53,5.98,18.158333,NaN,NaN,NaN,NaN,NaN,NaN
3,2021,Centro,55,3096,19583,19576,7,15.32,0.00,4935.90,5.57,6.56,16.375000,14.583333,17.139614,-5.510047,-9.821019,0.723327,9.698997
6,2022,Centro,58,3051,19945,19938,7,16.01,0.00,5466.32,5.78,6.61,18.225000,5.454545,-1.453488,10.746166,11.297710,3.770197,0.762195
9,2023,Centro,59,3109,20393,20385,8,16.77,0.00,5712.09,5.66,6.17,20.583333,1.724138,1.901016,4.496078,12.940101,-2.076125,-6.656581
12,2024,Centro,59,3268,20525,20518,7,17.70,0.00,6268.72,5.89,6.45,23.791667,0.000000,5.114185,9.744769,15.587045,4.063604,4.538088
1,2020,Tetuán,4,301,17586,17535,51,13.82,11.44,3648.00,5.34,4.53,15.750000,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,Tetuán,5,327,18140,18082,58,13.95,11.62,3756.93,5.24,4.48,14.425000,25.000000,8.637874,2.986020,-8.412698,-1.872659,-1.103753
7,2022,Tetuán,6,357,18764,18703,61,14.52,12.20,3955.42,4.89,4.09,15.658333,20.000000,9.174312,5.283303,8.549971,-6.679389,-8.705357
10,2023,Tetuán,9,399,19616,19549,67,15.22,12.52,4677.48,4.81,3.86,17.383333,50.000000,11.764706,18.254951,11.016498,-1.635992,-5.623472
13,2024,Tetuán,15,637,20337,20271,66,16.08,13.39,4755.80,4.69,3.78,19.825000,66.666667,59.649123,1.674406,14.046021,-2.494802,-2.072539


In [186]:
sns.lineplot(
    data=df_total,
    x="Año",
    y="plazas_por_1000_viviendas_alquiler",
    hue="Distrito",
    marker="o"
)

plt.title("Plazas turísticas por cada 1.000 viviendas en alquiler")
plt.show()

ValueError: Could not interpret value `plazas_por_1000_viviendas_alquiler` for `y`. An entry with this name does not appear in `data`.

In [288]:
df_total["plazas_turisticas_por_1000_hab"] = (
    df_total["Plazas en sstablecimientos turisticos"] / df_total["Poblacion"] * 1000
)

df_total["establecimientos_turisticos_por_1000_hab"] = (
    df_total["Establecimientos turisticos"] / df_total["Poblacion"] * 1000
)

df_total["viviendas_alquiler_por_1000_hab"] = (
    df_total["Nº viviendas en alquiler"] / df_total["Poblacion"] * 1000
)

df_total.head()

,Año,Distrito,Establecimientos turisticos,Plazas en sstablecimientos turisticos,Nº viviendas en alquiler,Nº viviendas en alquiler colectiva,Nº viviendas en alquiler unifamiliar,Renta vivienda colectiva (€/m².mes),Renta vivienda unifamiliar (€/m².mes),Precio vivienda (€/m²),Indice de Vulnerabilidad Territorial Agregado,Índice de Vulnerabilidad Bienestar Social e Igualdad,Precio alquiler €/m² media del año,Poblacion,plazas_turisticas_por_1000_hab,establecimientos_turisticos_por_1000_hab,viviendas_alquiler_por_1000_hab
0,2020,Centro,48,2643,18981,18974,7,15.36,0.00,5223.73,5.53,5.98,18.158333,140473,18.815004,0.341703,135.122052
1,2020,Tetuán,4,301,17586,17535,51,13.82,11.44,3648.00,5.34,4.53,15.750000,161313,1.865938,0.024797,109.017872
2,2020,Villaverde,1,1,7956,7887,69,8.65,8.12,1749.87,5.66,4.87,10.950000,154318,0.006480,0.006480,51.555878
3,2021,Centro,55,3096,19583,19576,7,15.32,0.00,4935.90,5.57,6.56,16.375000,141236,21.920757,0.389419,138.654451
4,2021,Tetuán,5,327,18140,18082,58,13.95,11.62,3756.93,5.24,4.48,14.425000,159849,2.045681,0.031280,113.482099


In [291]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
variables_cluster = [
    "Precio vivienda (€/m²)",
    "Precio alquiler €/m² media del año",
    "Indice de Vulnerabilidad Territorial Agregado",
    "plazas_turisticas_por_1000_hab",
    "establecimientos_turisticos_por_1000_hab"
]

X = df_total[variables_cluster].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [300]:
inertias = []

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

print(inertias)

[27.681032091787735, 8.854912803544089, 5.820580478307153, 3.376159918035211, 2.6058965990441947, 1.826986732749896]


In [301]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_total.loc[X.index, "cluster"] = kmeans.fit_predict(X_scaled)


perfil_clusters = df_total.groupby("cluster")[variables_cluster].mean().round(2)

print(perfil_clusters)

         Precio vivienda (€/m²)  Precio alquiler €/m² media del año  \
cluster                                                               
0.0                     1885.96                               11.95   
1.0                     5521.35                               19.43   
2.0                     4158.73                               16.61   

         Indice de Vulnerabilidad Territorial Agregado  \
cluster                                                  
0.0                                               5.70   
1.0                                               5.69   
2.0                                               4.99   

         plazas_turisticas_por_1000_hab  \
cluster                                   
0.0                                0.06   
1.0                               21.46   
2.0                                2.50   

         establecimientos_turisticos_por_1000_hab  
cluster                                            
0.0                              